# Synthetic data


In [2]:
import pandas as pd

# sample raw report
df = pd.read_csv("../data/raw/report.csv")
pd.set_option("display.max_rows", None)
display(df.T)

pd.reset_option("display.max_columns")
pd.reset_option("display.width")

,0
Organization,MBMR1
Point of Care,RAD MR MB
Source System,EPIC
Accession Number,20031798852
Modality,MRI
Exam Code,MBRUE
Exam Description,MR BRAIN WITH AND WITHOUT CONTRAST
CPT Code,70553
Report Text,MR BRAIN WITH AND WITHOUT CONTRAST: 5/15/2023...
Is Stat,False


## Synthetic data generation with OpenAI


In [10]:
import csv
import io
import json
import os
from typing import Any, Dict, List

import openai
import pandas as pd
from tqdm import tqdm

API_KEY = "YOUR_API_KEY"
API_VERSION = "2024-10-21"
RESOURCE_ENDPOINT = "https://YOUR_ENDPOINT.com"
MODEL = "gpt-4o-2024-08-06"

client = openai.AzureOpenAI(
    api_key=API_KEY,
    api_version=API_VERSION,
    azure_endpoint=RESOURCE_ENDPOINT,
)


def generate_synthetic_data(
    sample_data: pd.DataFrame, context_info: Dict[str, Any], num_samples: int = 2
) -> List[Dict[str, Any]]:
    """
    Generate synthetic data based on sample data while preserving privacy and data integrity.

    Args:
        sample_data (pd.DataFrame): Sample data to base synthetic data on
        context_info (Dict[str, Any]): Context information for generation (e.g., data_type, specific_context)
        num_samples (int): Number of synthetic samples to generate

    Returns:
        List[Dict[str, Any]]: List of dictionaries containing synthetic data records
    """
    data_type = context_info.get("data_type", "mri report")
    specific_context = context_info.get("specific_context", "")

    min_examples = 2
    real_examples = sample_data.sample(min(min_examples, len(sample_data))).to_dict(
        "records"
    )

    system_prompt = f"""You are a data synthesis expert specializing in medical {data_type} generation.
    Your task is to generate synthetic data that closely resembles the provided examples
    while ensuring complete privacy protection. The synthetic data should maintain the characteristics, 
    terminology, and structure of the real data but contain no actual identifiable information.
    
    Pay careful attention to the content of the example data to determine the appropriate medical domain and terminology.
    """

    user_prompt = f"""Given the following context: {specific_context}
    Generate {num_samples} synthetic {data_type} records based on these real examples:
    
    {json.dumps(real_examples, indent=2)}
    
    Requirements:
    1. Each record must follow the exact format of the examples
    2. Use realistic but synthetic identifiers
    3. Maintain the structure, length, and detail level of real data
    4. Carefully analyze the examples to identify the medical domain and use appropriate terminology
    5. ALL fields must have valid values - no empty fields
    
    Return your response in this exact JSON format:
    {{
    "records": [
        # {num_samples} records here with all fields
    ]
    }}
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.5,
        max_tokens=4000,
        response_format={"type": "json_object"},
    )

    # Add this debug code
    content_str = response.choices[0].message.content
    print(f"Raw API response for {data_type}:\n{content_str[:200]}...")

    try:
        # Parse response
        content = json.loads(content_str)

        # Debug the structure
        print(f"Response type: {type(content)}")
        print(
            f"Keys if dict: {content.keys() if isinstance(content, dict) else 'Not a dict'}"
        )

        # Extract records (handle different possible response formats)
        if isinstance(content, dict) and "records" in content:
            records = content["records"]
        elif isinstance(content, dict) and "data" in content:
            records = content["data"]
        elif isinstance(content, list):
            records = content
        else:
            # Try to find any list in the response
            for key, value in content.items():
                if isinstance(value, list) and len(value) > 0:
                    records = value
                    break
            else:
                records = [content]

        return records

    except Exception as e:
        print(f"Error generating synthetic data: {str(e)}")
        # Return empty list as fallback
        return []


sample = {
    "Organization": ["MBMR1", "MBMR1"],
    "Point of Care": ["RAD MR MB", "RAD MR MB"],
    "Source System": ["EPIC", "EPIC"],
    "Accession Number": ["2003179885", "2003179886"],
    "Patient MRN": ["12345678", "87654321"],
    "Report Text": [
        "FINDINGS: MRI of the brain demonstrates a 3.2 x 2.8 cm heterogeneously enhancing mass in the right temporal lobe with surrounding vasogenic edema. The mass demonstrates restricted diffusion and appears to involve the cortex and subcortical white matter. No midline shift or hydrocephalus.",
        "FINDINGS: MRI of the brain shows a 2.5 x 2.1 cm infiltrative lesion in the left frontal lobe with irregular enhancement pattern. There is moderate surrounding edema with no significant mass effect on the ventricles. No evidence of hemorrhage or calcification within the lesion.",
    ],
    "Exam Description": ["MRI BRAIN W AND WO CONTRAST", "MRI BRAIN W AND WO CONTRAST"],
    "Patient Status": ["Outpatient", "Inpatient"],
    "Total (RVU)": [12.41, 13.11],
    "Total (Professional) (RVU)": [5.65, 6.45],
    "Total (Technical) (RVU)": [6.76, 5.46],
}

synthetic = generate_synthetic_data(
    sample_data=pd.DataFrame(sample),
    context_info={"data_type": "MRI", "context_info": "brain MRI"},
    num_samples=1,
)

pd.DataFrame(synthetic)

Raw API response for MRI:
{
    "records": [
        {
            "Organization": "MBMR1",
            "Point of Care": "RAD MR MB",
            "Source System": "EPIC",
            "Accession Number": "2003179887",
         ...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])


,Organization,Point of Care,Source System,Accession Number,Patient MRN,Report Text,Exam Description,Patient Status,Total (RVU),Total (Professional) (RVU),Total (Technical) (RVU)
0,MBMR1,RAD MR MB,EPIC,2003179887,98765432,FINDINGS: MRI of the brain reveals a 2.9 x 2.4...,MRI BRAIN W AND WO CONTRAST,Inpatient,13.02,6.38,5.54


## Finetune data

In [7]:
def get_cancer_type(filename: str) -> str:
    """Extract and standardize cancer type from filename."""
    cancer_type = filename.replace("search_", "").split("_")[0]
    return cancer_type.lower()


# files in the system
CANCER_FILES = [
    "search_astrocytoma_MR_perReport.xlsx",
    "search_Ependymoma_MR_perReport.xlsx",
    "search_Ganglioglioma_MR_perReport.xlsx",
    "search_glioblastoma_MR_perReport.xlsx",
    "search_Gliomatosis_MR_perReport.xlsx",
    "search_Gliosarcoma_MR_perReport.xlsx",
    "search_oligoastrocytoma_MR_perReport.xlsx",
    "search_oligodendroglioma_MR_perReport.xlsx",
    "search_Xanthoastrocytoma_MR_perReport.xlsx",
]

COLUMNS_NEEDED = [
    "Patient MRN",
    "Report Text",
    "Exam Description",
    "Patient Status",
]

# Base directory with MRI and pathology folders
base_dir = "cancer_data"
mri_dir = os.path.join(base_dir, "mri")
path_dir = os.path.join(base_dir, "pathology")

os.makedirs(mri_dir, exist_ok=True)
os.makedirs(path_dir, exist_ok=True)


def process_synthetic_data(
    raw_records: List[Dict[str, Any]], sample_data: pd.DataFrame
) -> pd.DataFrame:
    """
    Process raw synthetic records into a DataFrame with the same structure as the sample data.

    Args:
        raw_records (List[Dict[str, Any]]): Raw synthetic records
        sample_data (pd.DataFrame): Original sample data with desired structure

    Returns:
        pd.DataFrame: Processed synthetic data
    """
    # Convert to DataFrame
    df = pd.DataFrame(raw_records)

    # Ensure all required columns exist
    for col in sample_data.columns:
        if col not in df.columns:
            df[col] = ""

    # Reorder columns to match original
    df = df[sample_data.columns]

    # Clean text fields
    if "Report Text" in df.columns:
        df["Report Text"] = df["Report Text"].apply(lambda x: str(x).strip())
    if "Exam Description" in df.columns:
        df["Exam Description"] = df["Exam Description"].apply(lambda x: str(x).strip())
    if "Patient MRN" in df.columns:
        df["Patient MRN"] = df["Patient MRN"].astype(str)

    return df


base_path = "/data/users/neob/mPower/perReport_12122024"

for filename in tqdm(CANCER_FILES, desc="Processing cancer files"):
    cancer_type = get_cancer_type(filename)
    print(f"\nProcessing {cancer_type}...")

    # Define file paths
    mri_file = f"{base_path}/{filename}"
    path_filename = filename.replace("_MR_", "_pathology_")
    path_file = f"{base_path}/{path_filename}"

    mri_output_path = f"{mri_dir}/{cancer_type}_synthetic.csv"
    path_output_path = f"{path_dir}/{cancer_type}_synthetic.csv"

    try:
        # Load data
        mri_df = pd.read_excel(mri_file, usecols=COLUMNS_NEEDED, nrows=100)
        path_df = pd.read_excel(path_file, usecols=COLUMNS_NEEDED, nrows=100)

        # Find common MRNs between MRI and pathology data
        mri_mrns = set(mri_df["Patient MRN"].astype(str))
        path_mrns = set(path_df["Patient MRN"].astype(str))
        common_mrns = list(mri_mrns.intersection(path_mrns))

        if len(common_mrns) < 2:
            print(
                f"Warning: Only {len(common_mrns)} common MRNs found for {cancer_type}. Using all available."
            )
            num_samples = min(2, len(common_mrns))
        else:
            num_samples = 2
            import random

            random.shuffle(common_mrns)
            common_mrns = common_mrns[:num_samples]

        print(f"Using {num_samples} common MRNs as reference: {common_mrns}")

        # Generate synthetic MRI data using matched reference data
        mri_synthetic_records = []
        path_synthetic_records = []

        # Create unique synthetic MRNs for each cancer type
        # Each cancer gets 2 unique MRNs (total 9*2=18 unique MRNs)
        cancer_index = CANCER_FILES.index(filename)
        start_mrn = 1000 + (cancer_index * 2)  # Ensure uniqueness across cancer types
        synthetic_mrns = [f"SYN{start_mrn + i}" for i in range(num_samples)]
        print(f"Created synthetic MRNs: {synthetic_mrns}")

        for i, real_mrn in enumerate(common_mrns):
            # Get reference data for this MRN
            mri_ref = mri_df[mri_df["Patient MRN"].astype(str) == real_mrn].iloc[0:1]
            path_ref = path_df[path_df["Patient MRN"].astype(str) == real_mrn].iloc[0:1]

            # Skip if either reference is empty
            if mri_ref.empty or path_ref.empty:
                continue

            synthetic_mrn = synthetic_mrns[i]

            # Generate synthetic MRI data with context from the matched reference
            mri_context = {
                "data_type": "mri",
                "specific_context": f"Brain MRI reports for {cancer_type} patients. These reports typically describe tumor location, size, enhancement patterns, and surrounding edema. {cancer_type} typically presents with specific imaging characteristics that should be reflected in the synthetic data.",
            }
            mri_records = generate_synthetic_data(mri_ref, mri_context, num_samples=1)

            if mri_records and len(mri_records) > 0:
                # Replace the MRN with our synthetic one
                mri_record = mri_records[0]
                mri_record["Patient MRN"] = synthetic_mrn
                mri_synthetic_records.append(mri_record)

            # Generate synthetic pathology data with context from the matched reference
            path_context = {
                "data_type": "pathology",
                "specific_context": f"Pathology reports for {cancer_type} brain tumors. These reports typically include histological features, immunohistochemistry results, molecular markers, and WHO grading specific to {cancer_type}. The synthetic data should maintain the specialized terminology and structure of neuropathology reports.",
            }
            path_records = generate_synthetic_data(
                path_ref, path_context, num_samples=1
            )

            if path_records and len(path_records) > 0:
                # Replace the MRN with our synthetic one (matching the MRI one)
                path_record = path_records[0]
                path_record["Patient MRN"] = synthetic_mrn
                path_synthetic_records.append(path_record)

        # Process into DataFrames with correct structure
        if mri_synthetic_records:
            synthetic_mri_df = process_synthetic_data(mri_synthetic_records, mri_df)
            synthetic_mri_df.to_csv(mri_output_path, index=False)
            print(
                f"Generated {len(synthetic_mri_df)} synthetic MRI records for {cancer_type}"
            )
            print(f"MRNs in MRI file: {synthetic_mri_df['Patient MRN'].tolist()}")
        else:
            print(f"No synthetic MRI records generated for {cancer_type}")

        if path_synthetic_records:
            synthetic_path_df = process_synthetic_data(path_synthetic_records, path_df)
            synthetic_path_df.to_csv(path_output_path, index=False)
            print(
                f"Generated {len(synthetic_path_df)} synthetic pathology records for {cancer_type}"
            )
            print(
                f"MRNs in pathology file: {synthetic_path_df['Patient MRN'].tolist()}"
            )
        else:
            print(f"No synthetic pathology records generated for {cancer_type}")

    except Exception as e:
        print(f"Error processing files for {cancer_type}: {str(e)}")

Processing cancer files:   0%|          | 0/9 [00:00<?, ?it/s]


Processing astrocytoma...
Using 2 common MRNs as reference: ['50475182', '50723143']
Created synthetic MRNs: ['SYN1000', 'SYN1001']
Raw API response for mri:
{
    "records": [
        {
            "Exam Description": "MR BRAIN WITH AND WITHOUT CONTRAST",
            "Report Text": "BRAIN MRI: 8/15/2023 10:30 AM\n\nCLINICAL HISTORY: Grade II diffuse astro...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])
Raw API response for pathology:
{
    "records": [
        {
            "Exam Description": "SURGICAL PATHOLOGY",
            "Report Text": "Anatomic Site: Left temporal lobe\nClinical Diagnosis/History:\nThe patient is a 10-year-...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])
Raw API response for mri:
{
    "records": [
        {
            "Exam Description": "MR BRAIN WITH AND WITHOUT CONTRAST",
            "Report Text": "BRAIN MRI: 3/15/2023 10:45 AM\n\nCLINICAL HISTORY: 72 year old man with r...
Response type: <class 'dict'>
Keys if

Processing cancer files:  11%|█         | 1/9 [00:20<02:43, 20.47s/it]

Raw API response for pathology:
{
    "records": [
        {
            "Exam Description": "SURGICAL PATHOLOGY",
            "Report Text": "Anatomic Site:Parietal lobe biopsy (FS)\nParietal Tumor (FS   perm)\nClinical Diagnosis/H...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])
Generated 2 synthetic MRI records for astrocytoma
MRNs in MRI file: ['SYN1000', 'SYN1001']
Generated 2 synthetic pathology records for astrocytoma
MRNs in pathology file: ['SYN1000', 'SYN1001']

Processing ependymoma...
Using 2 common MRNs as reference: ['46732251', '94140030']
Created synthetic MRNs: ['SYN1002', 'SYN1003']
Raw API response for mri:
{
    "records": [
        {
            "Exam Description": "MR BRAIN WITH AND WITHOUT CONTRAST",
            "Report Text": "MR BRAIN: 11/05/23\n\nCLINICAL HISTORY: Recurrent anaplastic ependymoma.\...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])
Raw API response for pathology:
{
    "records": [
        {
            "

Processing cancer files:  22%|██▏       | 2/9 [00:42<02:30, 21.52s/it]

...

Raw API response for pathology:
{
    "records": [
        {
            "Exam Description": "SURGICAL PATHOLOGY",
            "Report Text": "Anatomic Site: Left temporal cortex (FS)\nLeft temporal subcortical (FS)\nLeft temporal c...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])
Generated 2 synthetic MRI records for oligoastrocytoma
MRNs in MRI file: ['SYN1012', 'SYN1013']
Generated 2 synthetic pathology records for oligoastrocytoma
MRNs in pathology file: ['SYN1012', 'SYN1013']

Processing oligodendroglioma...
Using 2 common MRNs as reference: ['34907480', '36238798']
Created synthetic MRNs: ['SYN1014', 'SYN1015']
Raw API response for mri:
{
    "records": [
        {
            "Exam Description": "MR BRAIN WITH AND WITHOUT CONTRAST",
            "Report Text": "MR BRAIN WITH AND WITHOUT CONTRAST:  9/14/2023 2:45 PM\nINDICATION (as pr...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])
Raw API response for pathology:
{
    "records": [
      

Processing cancer files:  89%|████████▉ | 8/9 [03:00<00:21, 21.89s/it]

Raw API response for pathology:
{
    "records": [
        {
            "Exam Description": "SURGICAL PATHOLOGY",
            "Report Text": "Anatomic Site:Brain, biopsy (FS) - no req\nClinical Diagnosis/History:\nThe patient is a ...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])
Generated 2 synthetic MRI records for oligodendroglioma
MRNs in MRI file: ['SYN1014', 'SYN1015']
Generated 2 synthetic pathology records for oligodendroglioma
MRNs in pathology file: ['SYN1014', 'SYN1015']

Processing xanthoastrocytoma...
Using 2 common MRNs as reference: ['56044981', '12829388']
Created synthetic MRNs: ['SYN1016', 'SYN1017']
Raw API response for mri:
{
    "records": [
        {
            "Exam Description": "MR BRAIN WITH AND WITHOUT CONTRAST",
            "Report Text": "MR BRAIN WITH AND WITHOUT CONTRAST:  7/15/2023 9:30 AM\nINDICATION (as pr...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])
Raw API response for pathology:
{
    "records": [
    

Processing cancer files: 100%|██████████| 9/9 [03:38<00:00, 24.27s/it]

Raw API response for pathology:
{
    "records": [
        {
            "Exam Description": "SURGICAL PATHOLOGY",
            "Report Text": "UNIVERSITY OF SOUTHERN CALIFORNIA\nDEPARTMENT OF PATHOLOGY\n1234 HEALTH SCIENCES AVE., RO...
Response type: <class 'dict'>
Keys if dict: dict_keys(['records'])
Generated 2 synthetic MRI records for xanthoastrocytoma
MRNs in MRI file: ['SYN1016', 'SYN1017']
Generated 2 synthetic pathology records for xanthoastrocytoma
MRNs in pathology file: ['SYN1016', 'SYN1017']


## Eval data

In [ ]:
# replace with your path
tumor_type_eval_df = pd.read_csv("/home/neob/ai/paper/data/eval/tumor_type_eval.csv")

synthetic_tumor_type = generate_synthetic_data(
    sample_data=tumor_type_eval_df[["MRN", "report", "tumor_type"]].head(100),
    context_info={
        "data_type": "MRI",
        "specific_context": f"Tumor type evaluation dataset, cancer type must be one of the following {tumor_type_eval_df.tumor_type.unique()}",
    },
    num_samples=5,
)

pd.DataFrame(synthetic_tumor_type).to_csv("synthetic_tumor_type_eval.csv", index=False)

In [ ]:
# replace with your path
mgmt_eval_df = pd.read_csv("/home/neob/ai/paper/data/eval/mgmt_eval.csv")
synthetic_mgmt = generate_synthetic_data(
    sample_data=mgmt_eval_df[["mrn", "report", "mgmt"]].head(5),
    context_info={
        "data_type": "pathology",
        "specific_context": "MGMT status evaluation dataset",
    },
    num_samples=5,
)

pd.DataFrame(synthetic_mgmt).to_csv("synthetic_mgmt_eval.csv", index=False)

In [ ]:
# replace with your path
os_eval = pd.read_csv("/home/neob/ai/paper/data/eval/os_eval.csv")
synthetic_mgmt = generate_synthetic_data(
    sample_data=os_eval.head(100),
    context_info={
        "data_type": "MRI",
        "specific_context": "overall survival dataset of glioblastoma patients",
    },
    num_samples=2,
)

pd.DataFrame(synthetic_mgmt).to_csv("synthetic_os_eval.csv", index=False)